In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import pickle

# load index
index = faiss.read_index("/content/pubmedqa_index_flatl2.faiss")

# load mapping
with open("/content/chunk_mapping.pkl", "rb") as f:
    text_chunks = pickle.load(f)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
def retrieve(query, k=3):
    query_vector = model.encode([query])
    D, I = index.search(query_vector, k)
    return [text_chunks['question'].iloc[i] for i in I[0]]

In [ ]:
query = "What are the symptoms of diabetes?"

results = retrieve(query)

context = "\n".join(results)

print("Context:\n", context)

In [ ]:
!pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are a helpful medical assistant. Answer only based on the given context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]
)

print(response.choices[0].message.content)

In [ ]:
DISCLAIMER = "\n\n⚠️ This system is for educational purposes only."

answer = response.choices[0].message.content + DISCLAIMER
print(answer)

In [ ]:
log = {
    "question": query,
    "context": context,
    "answer": answer
}

print(log)